# 2. LCEL Chains (Runnables, pipe syntax)

The LangChain Expression Language (LCEL) lets you compose `Runnable`s with the `|`
operator. This notebook demonstrates:
- A simple linear pipe chain: `prompt | llm | output_parser`.
- `RunnableParallel`, running independent branches over the same input concurrently.
- `RunnableLambda`, a plain Python function lifted into the chain.
- `RunnablePassthrough`, carrying the original input through unchanged.

**Prerequisites:** Ollama running locally with `llama3.2` pulled.

### Setup

This cell makes the project's shared `tools`/`models` packages importable
regardless of where Jupyter's working directory actually is (it's usually
this notebook's own folder, not the repo root), and loads `.env` plus any
cached secrets in `.env.local` (populated by `scripts/lib/env.sh` the first
time you've run `scripts/start_app.sh` / `scripts/start_infra.sh`).

In [ ]:
import sys
from pathlib import Path

from dotenv import load_dotenv

project_root = Path.cwd()
while not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

load_dotenv(project_root / ".env")
load_dotenv(project_root / ".env.local", override=True)  # cached secrets, if resolve_env() has run at least once
print("Project root on sys.path:", project_root)

## Part 1 — a minimal pipe chain

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough

from models.chat_models.ollama_models import SupportedModel, get_chat_model

llm = get_chat_model(SupportedModel.llama3_2)

pipe_prompt = ChatPromptTemplate.from_messages(
    [("human", "Rewrite the following text so a 10-year-old could understand it:\n\n{text}")]
)
pipe_chain = pipe_prompt | llm | StrOutputParser()

original = "Retrieval-augmented generation combines a parametric language model with a non-parametric retrieval mechanism over an external corpus."
print(pipe_chain.invoke({"text": original}))

## Part 2 — fan-out with `RunnableParallel`

Two independent branches (a one-sentence summary, and a list of follow-up
questions) run over the same input, plus the original input carried through
unchanged via `RunnablePassthrough`.

In [ ]:
summary_prompt = ChatPromptTemplate.from_messages([("human", "Summarize the topic '{topic}' in one sentence.")])
questions_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "human",
            "List 3 follow-up questions about '{topic}', one per line, "
            "with no numbering or extra commentary.",
        )
    ]
)

to_prompt_input = RunnableLambda(lambda topic: {"topic": topic})
split_lines = RunnableLambda(lambda text: [line.strip("- ").strip() for line in text.splitlines() if line.strip()])

parallel_chain = RunnableParallel(
    summary=to_prompt_input | summary_prompt | llm | StrOutputParser(),
    questions=to_prompt_input | questions_prompt | llm | StrOutputParser() | split_lines,
    original_topic=RunnablePassthrough(),
)

# A plain string in -> each branch adapts it as needed; RunnablePassthrough
# hands the same string straight through unchanged.
result = parallel_chain.invoke("photosynthesis")
print("summary:", result["summary"])
print("questions:", result["questions"])
print("original_topic:", result["original_topic"])

## 🧪 Playground

**1. Add a third parallel branch** — e.g. a `"fun_fact"` branch with its own prompt.

In [ ]:
# TODO: add a fun_fact=... branch to RunnableParallel and re-invoke


**2. Change `split_lines`** to also number the questions, or to cap the list at 2 items.

In [ ]:
# TODO: modify split_lines and re-run Part 2


**3. Try a much broader topic** (e.g. `"the internet"`) — does the one-sentence summary constraint hold up?

In [ ]:
# TODO: try parallel_chain.invoke with a different topic
